# Lab 2. Notebook 02 — Feature Preprocessing and Transformation

**Course:** Artificial Intelligence with Python  
**Estimated time:** 60 minutes  
**Prerequisites:** Lab 1 and Lab 2 Notebook 01 (data quality assessment)

### Learning objectives

By the end of this notebook, you should be able to:

- distinguish **input features** from the **prediction target**;
- explain and demonstrate numerical **imputation**;
- standardize inconsistent categories and apply **one-hot encoding**;
- distinguish **standardization** from **min-max scaling**;
- recognize why learned preprocessing parameters must come from the **training set only**.

**Important:** This notebook introduces the building blocks using **small teaching examples**. We will combine them and safely fit them on the apartment **training data** in Notebook 03. The student exercises contain `TODO` cells; complete these yourself.

## 1. Reload the apartment dataset and identify features (8 min)

The CSV intentionally contains missing values, inconsistent categories, and some invalid measurements. Your cleaning work in Notebook 01 helped identify these issues.

**Notebook independence:** We reload the CSV here; no variables from Notebook 01 are required. Launch JupyterLab from the course repository root or open this notebook in `lab02/`, ensuring that `data/` is alongside the notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

path = Path("data/apartments_corrupted.csv")
if not path.exists():
    raise FileNotFoundError(
        f"Cannot find {path}. Open the notebook in lab02/ and keep data/ next to it."
    )

raw = pd.read_csv(path)
print("Dataset shape:", raw.shape)
display(raw.head())

Our future regression problem is to predict the **monthly rent**.

- **Target (`y`):** `monthly_rent_eur` — the value we want to predict.
- **Features (`X`):** apartment properties known when making a prediction.
- **Identifier:** `apartment_id` uniquely labels a record; it is generally *not* a useful predictor.

Unlike a feature, the target should **not** be filled in with a guessed rent. Missing-target rows need special handling before supervised training.

In [ ]:
target_column = "monthly_rent_eur"
id_column = "apartment_id"

feature_columns = [c for c in raw.columns if c not in (target_column, id_column)]
print("Features:", feature_columns)
print("Target:", target_column)
print("Rows missing target:", raw[target_column].isna().sum())

### Exercise 1 — Plan your feature types (3 min)

Create two Python lists named `numeric_features` and `categorical_features` from the seven input features. Identify which columns contain real numerical quantities, even if Pandas currently reports their dtype as `object`. Do not include `apartment_id` or `monthly_rent_eur`.

In [ ]:
# TODO: define numeric_features
# numeric_features = [ ... ]

# TODO: define categorical_features
# categorical_features = [ ... ]

# TODO: print both lists and verify that together they contain seven feature names

## 2. Numerical imputation (15 min)

Some algorithms cannot accept missing numerical values (`NaN`). **Imputation** fills those gaps using a rule learned from available observations. Typical strategies include **mean** and **median** imputation.

Consider these *illustrative observations*, not the entire apartment dataset. Their distribution is intentionally skewed by one large value.

In [ ]:
sample_areas = pd.Series([45.0, 50.0, 55.0, 500.0, np.nan], name="area_m2")
print("Mean:", sample_areas.mean())
print("Median:", sample_areas.median())
print("Fill with mean:", sample_areas.fillna(sample_areas.mean()).to_list())
print("Fill with median:", sample_areas.fillna(sample_areas.median()).to_list())

The unusually large area pulls the **mean** upward much more than the **median**. The median can be more robust to extreme values, although the best choice depends on the data.

In scikit-learn, `SimpleImputer` *learns* the replacement value during `fit()` and *applies* it during `transform()`.

In [ ]:
from sklearn.impute import SimpleImputer

# A tiny *training* example (not a split of the real apartment CSV).
train_example = pd.DataFrame({"area_m2": [45.0, 50.0, np.nan, 55.0]})
new_example = pd.DataFrame({"area_m2": [np.nan, 65.0]})

median_imputer = SimpleImputer(strategy="median")
median_imputer.fit(train_example)  # Learn only from example training observations

print("Learned median:", median_imputer.statistics_)
print("Transformed training data:")
print(median_imputer.transform(train_example))
print("Transformed new observations:")
print(median_imputer.transform(new_example))

**Key distinction:** `fit()` estimates something from data; `transform()` uses the previously learned estimate. If we calculate the median using the *test* set as well, we inadvertently allow future evaluation data to influence training. Notebook 03 will prevent this with a train/test split and a pipeline.

### Exercise 2 — Experiment with imputation (5 min)

Using the provided `ages_train` and `ages_new` DataFrames:

1. Create a `SimpleImputer(strategy="mean")` and fit it on `ages_train`.
2. Transform both `ages_train` and `ages_new` using the **same** fitted imputer.
3. Replace the mean strategy with `median`. What changes, and why?

**Expected check:** Neither transformed output should contain missing numerical values.

In [ ]:
ages_train = pd.DataFrame({"building_age_years": [5.0, 10.0, np.nan, 15.0, 70.0]})
ages_new = pd.DataFrame({"building_age_years": [np.nan, 30.0]})

# TODO: create and fit the mean imputer on ages_train
# TODO: transform ages_train and ages_new
# TODO: repeat with a median imputer, compare the results

## 3. Categorical features and one-hot encoding (15 min)

Many models operate on numbers, but our CSV contains categories: `district`, `balcony`, and `energy_rating`. First we should standardize the **meaning** of the categories (a cleaning operation); then we choose an encoding.

For nominal categories like `district`, assigning arbitrary integers such as `North = 1` and `South = 2` may suggest an order or distance that does not exist. **One-hot encoding** creates a separate indicator for each category.

In [ ]:
district_examples = pd.Series(["North", " north ", "NORTH", "South", np.nan], dtype="string")
normalized_districts = district_examples.str.strip().str.title()

print("Before:", district_examples.to_list())
print("After: ", normalized_districts.to_list())

A missing category can also be imputed. The most frequent category is a simple starting point, but for real projects you should consider whether missingness has meaning of its own.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Pedagogical example: a tiny training set and an unseen district.
train_districts = pd.DataFrame({"district": ["North", "South", "North", np.nan]})
new_districts = pd.DataFrame({"district": ["East", "North", np.nan]})

category_imputer = SimpleImputer(strategy="most_frequent")
train_filled = category_imputer.fit_transform(train_districts)
new_filled = category_imputer.transform(new_districts)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_encoded = encoder.fit_transform(train_filled)
new_encoded = encoder.transform(new_filled)

print("Categories learned:", encoder.categories_)
print("Feature names:", encoder.get_feature_names_out(["district"]).tolist())
print("Encoded training rows:", train_encoded)
print("Encoded new rows:", new_encoded)

**Notice:** `East` never appeared in the toy training data. With `handle_unknown="ignore"`, its encoded indicators are all zero rather than causing an error. This is practical, but an all-zero row must not be mistaken for a genuine observed training category.

**Do not `fit()` the encoder on the full apartment CSV yet.** The category vocabulary is another learned property, so it belongs to the training data only.

### Exercise 3 — Encode the balcony feature (5 min)

Using the **toy** DataFrames below:

1. Normalize `YES`, ` yes `, `No`, and `n` to a consistent `yes`/`no` vocabulary.
2. Fit an `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` on the normalized training values.
3. Transform the normalized new observations using the same encoder.
4. Inspect the resulting feature names.

*Hint:* `.astype("string").str.strip().str.lower().replace({"n": "no"})` works well here.

In [ ]:
balcony_train = pd.DataFrame({"balcony": ["YES", " no ", "yes", "n"]})
balcony_new = pd.DataFrame({"balcony": ["Yes", "No"]})

# TODO: normalize text in both DataFrames
# TODO: fit a OneHotEncoder on balcony_train only
# TODO: transform balcony_new and display the results

## 4. Feature scaling (14 min)

The size of a numerical feature can affect models based on distances or regularization. Compare apartment **area in square metres** (tens or hundreds) with **distance to the city centre in kilometres** (usually much smaller).

Two popular transformations:

- **StandardScaler:** subtracts the training mean, divides by the training standard deviation; transformed values need not be between 0 and 1.
- **MinMaxScaler:** maps the training minimum and maximum to 0 and 1; *new* observations can lie outside this range.

For tree-based models, scaling is often unnecessary. These transformations are tools, not obligatory steps for every algorithm.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# A small, explicit training example: each column is one feature.
scale_train = pd.DataFrame({
    "area_m2": [40.0, 60.0, 80.0],
    "distance_to_center_km": [2.0, 6.0, 10.0]
})
scale_new = pd.DataFrame({
    "area_m2": [100.0],
    "distance_to_center_km": [14.0]
})

standard_scaler = StandardScaler().fit(scale_train)
minmax_scaler = MinMaxScaler().fit(scale_train)

print("Original training rows:", scale_train)
print("Standardized training rows:", standard_scaler.transform(scale_train))
print("Min-max training rows:", minmax_scaler.transform(scale_train))
print("New observation, min-max scaled:", minmax_scaler.transform(scale_new))

Note that the new observation is outside the training range. The MinMaxScaler result exceeds 1: this is *expected*. Never refit a scaler independently on each new observation or test set.

### Exercise 4 — Compare scaled features (5 min)

Using the **toy** `scale_train` data:

1. Fit a new `StandardScaler`.
2. Create a DataFrame of the standardized values with the original column names.
3. Print each transformed feature's mean (approximately 0).
4. Print each feature's standard deviation with `ddof=0` (approximately 1).

**Question:** Why do we explicitly use `ddof=0` in this check?

In [ ]:
# TODO: fit StandardScaler on scale_train
# TODO: create a DataFrame containing the scaled values
# TODO: inspect column means and population standard deviations (ddof=0)

## 5. Apply the *preparation rules* to apartment data (8 min)

Before using scikit-learn transformers on the real dataset, we need reliable feature types and consistent category labels. We can safely perform **row-wise, rule-based corrections** on the full dataset because these rules do not estimate statistics from other rows.

The following demonstration converts numerical text and normalizes known category variants. It does *not* impute missing values, compute scaling statistics, or fit an encoder.

In [ ]:
apartment_features = raw.drop(columns=["apartment_id", "monthly_rent_eur"]).copy()

numeric_cols = ["area_m2", "rooms", "building_age_years", "distance_to_center_km"]
category_cols = ["district", "balcony", "energy_rating"]

for column in numeric_cols:
    apartment_features[column] = pd.to_numeric(apartment_features[column], errors="coerce")

# Row-wise corrections according to known plausibility rules.
apartment_features.loc[apartment_features["area_m2"] <= 0, "area_m2"] = np.nan
apartment_features.loc[apartment_features["rooms"] < 1, "rooms"] = np.nan
apartment_features.loc[apartment_features["building_age_years"] < 0, "building_age_years"] = np.nan
apartment_features.loc[apartment_features["distance_to_center_km"] < 0, "distance_to_center_km"] = np.nan

apartment_features["district"] = apartment_features["district"].astype("string").str.strip().str.title()
apartment_features["balcony"] = (
    apartment_features["balcony"].astype("string").str.strip().str.lower()
    .replace({"true": "yes", "n": "no"})
)
apartment_features["energy_rating"] = apartment_features["energy_rating"].astype("string").str.strip().str.upper()

print("Feature types after row-wise corrections:")
print(apartment_features.dtypes)
print("Missing features (not filled yet):")
print(apartment_features.isna().sum())
display(apartment_features.head())

**Important for the next notebook:** This DataFrame is **not yet ML-ready**. It still contains missing values and text categories. Some extreme values may require manual review, and we have not decided how to handle duplicate IDs or rows without target labels. Once we create training and testing datasets, the imputer, scaler, and encoder must be fitted on **training data only**.

### Mini-challenge — Make a preprocessing plan (4 min)

Write a short recommendation for predicting `monthly_rent_eur` using the apartment features:

1. Which **four numerical features** should receive imputation? Which scaling method would you try first for a distance-based algorithm?
2. Which **three categorical features** need encoding? How would you handle missing categories?
3. Should `apartment_id` be included as a predictor? Why or why not?
4. Why is it a mistake to fit a scaler or imputer on the entire CSV *before* splitting the data?

Write your answers in the Markdown cell below. You do not need to train a model.

**Your preprocessing plan:**

- Numerical features and imputation: *TODO*
- Categorical features and encoding: *TODO*
- Identifier/target decisions: *TODO*
- Leakage prevention: *TODO*

## Wrap-up · Ready for Notebook 03

You have learned the core scikit-learn transformations:

| Goal | Typical tool | What `fit()` learns |
|---|---|---|
| Fill numeric gaps | `SimpleImputer(strategy="median")` | Median of each numeric feature |
| Fill category gaps | `SimpleImputer(strategy="most_frequent")` | Most frequent category |
| Encode categories | `OneHotEncoder(handle_unknown="ignore")` | Category vocabulary |
| Standardize features | `StandardScaler()` | Mean and standard deviation |
| Scale to a range | `MinMaxScaler()` | Minimum and maximum |

**Next:** `03-preprocessing-pipelines.ipynb` will split the data, construct a `ColumnTransformer` and `Pipeline`, and fit every learned transformation *only on the training partition*. This is essential before Lab 3 model training.